# Content-Based TED Talks Recommendation Engine

In [1]:
import difflib

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv("../data/processed/ted_talks_cleaned.csv")
df["combined_text"] = df["combined_text"].fillna("")
df = df.reset_index(drop=True)

print("Talks loaded:", len(df))

Talks loaded: 4005


In [3]:
vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(df["combined_text"])

similarity_matrix = cosine_similarity(tfidf_matrix)

print("TF-IDF shape:", tfidf_matrix.shape)
print("Similarity table shape:", similarity_matrix.shape)

TF-IDF shape: (4005, 71129)
Similarity table shape: (4005, 4005)


In [4]:
# Map each title (lowercase, trimmed) to its row number
title_to_index = pd.Series(df.index, index=df["title"].str.lower().str.strip())
title_to_index = title_to_index[~title_to_index.index.duplicated()]

print("Titles in lookup:", len(title_to_index))

Titles in lookup: 4005


In [5]:
print("Talk 0 vs itself:", round(similarity_matrix[0, 0], 3))
print("Row number for 'do schools kill creativity?':", title_to_index["do schools kill creativity?"])

Talk 0 vs itself: 1.0
Row number for 'do schools kill creativity?': 4


In [6]:
lower_titles = list(title_to_index.index)


def suggest_titles(text, n=3):
    """Return up to n real talk titles that look close to what was typed."""
    matches = difflib.get_close_matches(text.lower().strip(), lower_titles, n=n, cutoff=0.5)
    return [df.loc[title_to_index[m], "title"] for m in matches]

In [7]:
def recommend_talks(title, top_n=5):
    """Return the top_n talks most similar to the given title as a DataFrame.

    If the title is not found, print close matches and return None.
    """
    # 1. Find the talk's row number
    if not isinstance(title, str) or not title.strip():
        print("Please enter a talk title as text.")
        return None

    idx = title_to_index.get(title.lower().strip())

    if idx is None:
        print(f'"{title}" was not found in the dataset.')
        suggestions = suggest_titles(title)
        if suggestions:
            print("Did you mean:")
            for s in suggestions:
                print("  -", s)
        return None

    # 2. Take this talk's row from the similarity table
    scores = similarity_matrix[idx].copy()

    # 3. Make sure the talk can't recommend itself
    scores[idx] = -1

    # 4. Pick the highest scores
    top_idx = np.argsort(scores)[::-1][:top_n]

    # 5. Build a readable table
    return pd.DataFrame({
        "Rank": range(1, len(top_idx) + 1),
        "Title": df.loc[top_idx, "title"].values,
        "Speaker": df.loc[top_idx, "speaker_1"].values,
        "Similarity": scores[top_idx].round(3),
    })

In [8]:
recommend_talks("Do schools kill creativity?")

,Rank,Title,Speaker,Similarity
0,1,How to escape education's death valley,Sir Ken Robinson,0.322
1,2,Bring on the learning revolution!,Sir Ken Robinson,0.296
2,3,How I stopped the Taliban from shutting down m...,Sakena Yacoobi,0.258
3,4,Our failing schools. Enough is enough!,Geoffrey Canada,0.257
4,5,How to run a company with (almost) no rules,Ricardo Semler,0.249


In [9]:
recommend_talks("A new strategy in the war on cancer")

,Rank,Title,Speaker,Similarity
0,1,A simple new blood test that can catch cancer ...,Jimmy Lin,0.606
1,2,Understanding cancer through proteomics,Danny Hillis,0.554
2,3,Open-source cancer research,Jay Bradner,0.525
3,4,New nanotech to detect cancer early,Joshua Smith,0.516
4,5,The future of early cancer detection?,Jorge Soto,0.475


In [10]:
recommend_talks("averting the climate crisis", top_n=3)

,Rank,Title,Speaker,Similarity
0,1,New thinking on the climate crisis,Al Gore,0.266
1,2,How we could change the planet's climate future,David Wallace-Wells,0.244
2,3,A climate solution where all sides can win,Ted Halstead,0.238


In [11]:
recommend_talks("Do school kill creativity")

"Do school kill creativity" was not found in the dataset.
Did you mean:
  - Do schools kill creativity?
  - Laws that choke creativity
  - 4 lessons in creativity


In [12]:
recommend_talks("zzz not a real talk")

"zzz not a real talk" was not found in the dataset.
Did you mean:
  - Is there a real you?


In [13]:
test_titles = [
    "A new strategy in the war on cancer",
    "averting the climate crisis",
    "Do schools kill creativity?",
]

for title in test_titles:
    print("Selected talk:", title)
    display(recommend_talks(title))
    print()

Selected talk: A new strategy in the war on cancer


,Rank,Title,Speaker,Similarity
0,1,A simple new blood test that can catch cancer ...,Jimmy Lin,0.606
1,2,Understanding cancer through proteomics,Danny Hillis,0.554
2,3,Open-source cancer research,Jay Bradner,0.525
3,4,New nanotech to detect cancer early,Joshua Smith,0.516
4,5,The future of early cancer detection?,Jorge Soto,0.475



Selected talk: averting the climate crisis


,Rank,Title,Speaker,Similarity
0,1,New thinking on the climate crisis,Al Gore,0.266
1,2,How we could change the planet's climate future,David Wallace-Wells,0.244
2,3,A climate solution where all sides can win,Ted Halstead,0.238
3,4,Climate change is happening. Here's how we adapt,Alice Bows-Larkin,0.235
4,5,Can we stop climate change by removing CO2 fro...,Tim Kruger,0.231



Selected talk: Do schools kill creativity?


,Rank,Title,Speaker,Similarity
0,1,How to escape education's death valley,Sir Ken Robinson,0.322
1,2,Bring on the learning revolution!,Sir Ken Robinson,0.296
2,3,How I stopped the Taliban from shutting down m...,Sakena Yacoobi,0.258
3,4,Our failing schools. Enough is enough!,Geoffrey Canada,0.257
4,5,How to run a company with (almost) no rules,Ricardo Semler,0.249


In [14]:
recommend_talks("averting the climate crisis", top_n=3)

,Rank,Title,Speaker,Similarity
0,1,New thinking on the climate crisis,Al Gore,0.266
1,2,How we could change the planet's climate future,David Wallace-Wells,0.244
2,3,A climate solution where all sides can win,Ted Halstead,0.238


In [18]:
random_title = df["title"].sample(1, random_state=7).iloc[0]
print("Selected talk:", random_title)
recommend_talks(random_title)

Selected talk: How your "working memory" makes sense of the world


,Rank,Title,Speaker,Similarity
0,1,A mouse. A laser beam. A manipulated memory.,Steve Ramirez,0.346
1,2,Feats of memory anyone can do,Joshua Foer,0.343
2,3,How memories form and how we lose them,Catharine Young,0.325
3,4,How computer memory works,Kanawat Senanan,0.303
4,5,How reliable is your memory?,Elizabeth Loftus,0.283


In [16]:
recommend_talks("Do school kill creativity")
print()
recommend_talks("zzz not a real talk")
print()
recommend_talks("")
print()
recommend_talks(123)


"Do school kill creativity" was not found in the dataset.
Did you mean:
  - Do schools kill creativity?
  - Laws that choke creativity
  - 4 lessons in creativity

"zzz not a real talk" was not found in the dataset.
Did you mean:
  - Is there a real you?

Please enter a talk title as text.

Please enter a talk title as text.


In [17]:
import time

start = time.time()
recommend_talks("Do schools kill creativity?")
print("Time taken:", round(time.time() - start, 4), "seconds")

Time taken: 0.0021 seconds


## Observations

1. **The function matches my Task 4 results.** For "Do schools kill creativity?" it returned the same five talks with the same scores (0.322 to 0.249), so the reusable version works correctly.
2. **The selected talk never recommends itself.** Its own score is 1.0, so the function sets it to -1 before sorting.
3. **The function is fast because the heavy work is done once.** TF-IDF and the similarity table took several seconds to build (7.7 seconds in Cell 3). After that, each call only looks up one row, so it takes a tiny fraction of a second.
4. **Typing style does not matter.** Capital letters and extra spaces are ignored, so "averting the climate crisis" works as well as the exact title.
5. **Wrong titles are handled without crashing.** A misspelled title gives "Did you mean" suggestions. A nonsense, empty or non-text input gives a clear message.
6. **Scores differ between talks.** Cancer talks scored higher (about 0.48 to 0.61) than climate talks (about 0.23 to 0.27), even though all results were relevant. So scores are only useful for ranking one talk's results.

## Limitations

- The method matches words, not meaning. "How to run a company with (almost) no rules" was recommended for a schools talk, probably because of shared vocabulary.
- Rare words and names can raise scores without describing the topic.
- It uses only the talk's text. It knows nothing about the viewer's taste, so it cannot give personalised results yet.
- A new talk is not in the table. The TF-IDF and similarity table would have to be rebuilt to include it.
- A high score does not guarantee that a viewer will like the recommendation.

## Review Questions

**1. Why do we convert text to TF-IDF vectors?**
A computer cannot compare sentences, but it can compare numbers. TF-IDF turns each talk into numbers, giving higher scores to words that are special to that talk and lower scores to words that appear everywhere.

**2. Why use cosine similarity?**
It measures how closely two talks point in the same direction, regardless of how long the talks are. A score near 1 means very similar and a score near 0 means almost nothing in common.

**3. Why should a talk not recommend itself?**
Its similarity with itself is always 1.0, so it would always come first and tell the viewer nothing.

**4. Why do we calculate the similarity matrix only once?**
Building it takes several seconds because it compares all 4,005 talks with each other. Once it exists, each recommendation is a quick lookup, so the function stays fast.

**5. How does the function handle invalid titles?**
It ignores capital letters and extra spaces. If the title is still not found, it prints close matches ("Did you mean") and returns nothing instead of crashing.

**6. What does a similarity score of 0.3 mean?**
It is not a percentage. It only says how much the important words overlap, and it is only meaningful for ranking the results of one talk.

**7. What are the limitations of this approach?**
It matches words, not meaning, it ignores the viewer's taste, and new talks need the table to be rebuilt.

**8. How could it be improved?**
Add word pairs (bigrams), try meaning-based embeddings, weight the topics more heavily, or combine it with viewer behaviour data.